# STEP 4b — Train the RadioUNet-style surrogate on Colab

Clones `cgm2179/indoor-walk-test`, generates (floor + Tx → path-loss map) pairs with the STEP_2 Motley-Keenan engine, and trains a U-Net to predict heatmaps in milliseconds.

**Before running:** Runtime → Change runtime type → **GPU (T4)**.

**Private repo:** the clone cell will prompt for a GitHub token. Create one at GitHub → Settings → Developer settings → Fine-grained tokens → repo `indoor-walk-test`, permission *Contents: Read*. (Or make the repo public and no token is needed.)

In [ ]:
import os
REPO = 'github.com/cgm2179/indoor-walk-test.git'
if not os.path.exists('indoor-walk-test'):
    if os.system(f'git clone https://{REPO}') != 0:   # private -> needs token
        from getpass import getpass
        token = getpass('GitHub fine-grained token: ')
        assert os.system(f'git clone https://{token}@{REPO}') == 0, 'clone failed'
%cd indoor-walk-test

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
# ~6 min on the Colab CPU; deterministic (fixed seed)
!python STEP_4/generate_dataset.py --n-samples 400

In [ ]:
import numpy as np
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

X_SCALE = np.array([20.0, 0.3, 3.0], np.float32).reshape(3, 1, 1)
Y_SCALE = 150.0                     # PL dB -> ~[0.3, 1]
CROP_H, CROP_W = 256, 568           # divisible by 8 for 3 poolings

class PLDataset(Dataset):
    def __init__(self, files):
        self.files = files
    def __len__(self):
        return len(self.files)
    def __getitem__(self, i):
        d = np.load(self.files[i])
        x = (d['x'][:, :CROP_H, :CROP_W] / X_SCALE).astype(np.float32)
        y = d['y'][:CROP_H, :CROP_W].astype(np.float32) / Y_SCALE
        m = d['indoor'][:CROP_H, :CROP_W]
        return torch.from_numpy(x), torch.from_numpy(y), torch.from_numpy(m)

files = sorted(Path('STEP_4/dataset').glob('sample_*.npz'))
split = int(0.85 * len(files))
train_dl = DataLoader(PLDataset(files[:split]), batch_size=8, shuffle=True, num_workers=2)
val_dl = DataLoader(PLDataset(files[split:]), batch_size=8, num_workers=2)
print(f'{split} train / {len(files) - split} val samples')

In [ ]:
import torch.nn as nn

def block(i, o):
    return nn.Sequential(
        nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
        nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True))

class UNet(nn.Module):
    def __init__(self, ch=(32, 64, 128, 256)):
        super().__init__()
        self.enc = nn.ModuleList()
        prev = 3
        for c in ch:
            self.enc.append(block(prev, c)); prev = c
        self.pool = nn.MaxPool2d(2)
        self.up, self.dec = nn.ModuleList(), nn.ModuleList()
        for c in reversed(ch[:-1]):
            self.up.append(nn.ConvTranspose2d(prev, c, 2, stride=2))
            self.dec.append(block(2 * c, c)); prev = c
        self.head = nn.Conv2d(prev, 1, 1)

    def forward(self, x):
        skips = []
        for i, e in enumerate(self.enc):
            x = e(x)
            if i < len(self.enc) - 1:
                skips.append(x)
                x = self.pool(x)
        for u, d in zip(self.up, self.dec):
            x = d(torch.cat([u(x), skips.pop()], 1))
        return self.head(x)[:, 0]

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
net = UNet().to(dev)
sum(p.numel() for p in net.parameters()) / 1e6

In [ ]:
EPOCHS = 30
opt = torch.optim.Adam(net.parameters(), 1e-3)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

def run(dl, train):
    net.train(train)
    tot = n = 0
    with torch.set_grad_enabled(train):
        for x, y, m in dl:
            x, y, m = x.to(dev), y.to(dev), m.to(dev)
            loss = ((net(x) - y).abs() * m).sum() / m.sum()   # indoor-masked L1
            if train:
                opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * x.size(0); n += x.size(0)
    return tot / n * Y_SCALE            # mean abs error, dB

for ep in range(EPOCHS):
    tr, va = run(train_dl, True), run(val_dl, False)
    sched.step()
    print(f'epoch {ep + 1:2d}   train MAE {tr:5.2f} dB   val MAE {va:5.2f} dB')

In [ ]:
import matplotlib.pyplot as plt
x, y, m = next(iter(val_dl))
with torch.no_grad():
    p = net(x.to(dev)).cpu()

rows = min(3, x.shape[0])
fig, axes = plt.subplots(rows, 3, figsize=(16, 2.6 * rows), squeeze=False)
for j in range(rows):
    panels = [(y[j], 'simulated PL (dB)', 'turbo', 40, 140),
              (p[j], 'U-Net prediction', 'turbo', 40, 140),
              ((p[j] - y[j]).abs(), '|error| (dB)', 'magma', 0, 15)]
    for k, (img, ttl, cm, lo, hi) in enumerate(panels):
        a = axes[j][k]
        shown = np.where(m[j], img.numpy() * Y_SCALE, np.nan)
        im = a.imshow(shown, cmap=cm, vmin=lo, vmax=hi)
        a.set_title(ttl if j == 0 else '', fontsize=10)
        a.axis('off')
        fig.colorbar(im, ax=a, shrink=0.75)
plt.tight_layout()

In [ ]:
torch.save(net.state_dict(), 'surrogate_unet.pt')
print('saved surrogate_unet.pt')
# keep it past the Colab session -- either download:
# from google.colab import files; files.download('surrogate_unet.pt')
# or copy to Drive:
# from google.colab import drive; drive.mount('/content/drive')
# !cp surrogate_unet.pt /content/drive/MyDrive/